# Data Engineering and Management Workshop

This workbook supports the presentation for the workshop. Use it as the working copy while the slides introduce the main ideas.

The case study follows a fictional prosthetics charity and shows the core parts of a data engineering workflow: ingestion, cleaning, joins, and privacy masking.

You will work through four themes:

- Ingestion and pipelines
- Data cleaning
- Cybersecurity and privacy masking
- Relational storage and joins

The dataset `prosthetics_data.csv` is already in this folder — you do not need to run any script to generate it.

If you want a small worked pattern for a step, open `snippets.ipynb` in this folder and adapt the example that matches the task.

## How to use this workbook

Read the short explanation above each section first, then fill in the code cell below it. If a step is unfamiliar, use the matching example in `snippets.ipynb` and keep only the part you need. If you already know the pattern, just complete the task directly.

---
## Step 0 — Setup

In [1]:
from pathlib import Path
import re
import hashlib
import pandas as pd

# The CSV is in this folder or the parent.
# Load it into a DataFrame, then inspect it:
#   df.head()        — first few rows, shows the format variety immediately
#   df.info()        — column types and non-null counts; messy types appear as 'object'
#   df.isna().sum()  — which columns have missing values
raise NotImplementedError('Load prosthetics_data.csv and run the three inspection calls.')

NotImplementedError: Load prosthetics_data.csv and run the three inspection calls.

---
## Section 1 — Ingestion & Pipelines (ETL vs ELT)

ETL means **Extract, Transform, Load**. Data is validated before it reaches the destination system. In a healthcare setting that usually means privacy checks and validation happen before sensitive records move anywhere.

ELT means **Extract, Load, Transform**. Raw data lands first and is transformed later inside a warehouse. That works well at scale, but raw patient records sit exposed until the transform runs.

This workshop follows the ETL path: clean and mask before anything leaves the secure environment.

---
## Section 2 — Data Cleaning

Three cleaning problems in this dataset: messy ages, inconsistent date formats, and inconsistent casing on categories. One function each, so each piece is independently testable.

### 2a — Parse Age_Input

The column contains plain ages, birth years, and freeform text like `'around 50'`. Write a function that normalises all three to a clean integer and stores the result in `Age_Years`.

- Missing values → `None`
- Number ≤ 100 → already an age, return as-is
- Number > 1900 → birth year, subtract from 2026
- Text → extract the first number you find

In [ ]:
CURRENT_YEAR = 2026

def parse_age_input(value):
    # Check pd.isna(value) first — return None if missing.
    # If value is already an int or float, convert with int(value) directly.
    # Otherwise use re.search(r'\d+', str(value)) to pull the first digit run from text.
    # Numbers <= 100 are already an age; numbers > 1900 are birth years — subtract from CURRENT_YEAR.
    raise NotImplementedError('Implement parse_age_input.')

df['Age_Years'] = df['Age_Input'].apply(parse_age_input)
df[['Patient_ID', 'Age_Input', 'Age_Years']]

### 2b — Parse Implant_Date

The column has several different date formats. A single `pd.to_datetime(dayfirst=True)` call is not safe here: it also reorders day and month inside year-first strings like `2023-09-01`, silently turning September into January.

Fix: detect year-first values with a regex and convert them directly to ISO format, then let pandas handle named months and `DD/MM/YYYY` with `dayfirst=True`. Strip ordinal suffixes like `'12th'` → `'12'` first so named-month dates parse cleanly.

Store results as `YYYY-MM-DD` strings in `Implant_Date_Clean`.

In [ ]:
def parse_implant_date(raw):
    # Hint: strip ordinal suffixes first — re.sub(r'(\d+)(st|nd|rd|th)', r'\1', ...)
    # Then detect year-first format with: re.match(r'^(\d{4})[-/](\d{1,2})[-/](\d{1,2})$', s)
    # If matched, build the ISO string directly — don't let pandas touch it.
    # For everything else: pd.to_datetime(s, dayfirst=True).strftime('%Y-%m-%d')
    raise NotImplementedError('Implement parse_implant_date.')

df['Implant_Date_Clean'] = df['Implant_Date'].apply(parse_implant_date)
df[['Patient_ID', 'Implant_Date', 'Implant_Date_Clean']]

### 2c — Standardise Amputation_Cause

Values are correct but inconsistently cased and sometimes padded with spaces. A chained `.str.lower().str.strip()` fixes both. Run `.value_counts()` after — one entry will still look wrong, and that's worth noticing.

In [ ]:
df['Amputation_Cause_Clean'] = df['Amputation_Cause'].str.lower().str.strip()
df['Amputation_Cause_Clean'].value_counts()

## Section 3: Cybersecurity & Cloud Privacy (PII Masking)

PII means Personally Identifiable Information. Names, contact details, and other identifiers should be protected, minimised, or removed before data is shared or uploaded.

Before patient data is moved to public cloud storage or joined with external datasets, it should be anonymised or pseudonymised. Hashing can replace names with stable tokens that do not expose the original value. This is part of the Transform phase in ETL.

In [ ]:
# Write a function that anonymises Full_Name using SHA-256 from hashlib.
# Apply this right after data cleaning and before the join with device inventory.
# This ensures sensitive identifiers are masked as part of the ETL transformation phase.
# Example starting point:
# import hashlib
# def hash_name(value):
#     normalized_value = str(value).strip().lower()
#     return hashlib.sha256(normalized_value.encode('utf-8')).hexdigest()
# df['Full_Name'] = df['Full_Name'].apply(hash_name)
raise NotImplementedError('Participant task: hash the Full_Name column with SHA-256.')

---
## Section 4 — Relational Storage & Joins

Structured data fits into tables, rows, and columns, so SQL databases are a good fit when you need rules and relationships. Unstructured data includes notes, messages, and scanned documents.

Primary keys identify rows. Foreign keys point to related rows in other tables. Joins use those keys to connect data without duplicating it.

For this charity, patient records can be joined to device inventory so analysts can see which model was fitted, what it cost, and who made it. Privacy-masked identifiers are now safe to join and share.

In [ ]:
device_inventory = {
    'M210': {'Model_Name': 'AeroStep Lite', 'Cost_USD': 950, 'Manufacturer': 'Stride MedTech'},
    'M302': {'Model_Name': 'FlexFoot Alpha', 'Cost_USD': 1200, 'Manufacturer': 'Global Limb Works'},
    'M404': {'Model_Name': 'CrestFit Pro', 'Cost_USD': 1850, 'Manufacturer': 'OrthoSphere'},
    'M505': {'Model_Name': 'AdaptMax Field', 'Cost_USD': 2100, 'Manufacturer': 'Humanity Mobility Systems'},
}
# Convert the inventory dict to a DataFrame first, then join it to the patient data.
# device_inventory_df = pd.DataFrame.from_dict(device_inventory, orient='index').reset_index()
# device_inventory_df = device_inventory_df.rename(columns={'index': 'Device_Model_ID'})
# merged_df = pd.merge(df, device_inventory_df, on='Device_Model_ID', how='inner')
raise NotImplementedError('Participant task: merge the patient and device inventory data.')

## Wrap-Up

By the end of the exercises, you should be able to explain how messy inputs are handled, how joins connect business tables, and why privacy controls matter before cloud upload.

If you wanted a smaller starting pattern, check `snippets.ipynb` and adapt it to this workbook.